# Air Quality Forecasting Pegasus Workflow

Air quality monitoring is critical for public health and environmental research. The [OpenAQ](https://openaq.org/) platform aggregates real-time and historical air quality data from government-grade monitoring stations around the world, making it freely accessible via an open API.

This workflow fetches air quality measurements (PM2.5, PM10, O3, NO2, SO2, CO) from OpenAQ monitoring stations, computes EPA-standard **Air Quality Index (AQI)** values, and uses an **LSTM neural network** to generate 24-hour AQI forecasts.

The workflow has two parallel pipelines that run after data extraction:

**Base Pipeline:** Analysis and anomaly detection on the extracted time series data.

**Forecast Pipeline:** Historical data fetching, feature engineering, LSTM model training, forecast generation, and visualization.

## Containers

All tools required to execute the jobs are included in a single Apptainer container built from the definition file in this repository:

`Apptainer/AirQuality_Forecast_Container.def`, built to a `.sif` that Pegasus stages to the worker nodes (no registry pull), with:
* pandas, numpy, matplotlib, scipy, requests (base analysis)
* torch, scikit-learn, tqdm (ML forecasting)
* sage-data-client (SAGE sensor integration)

## Accessing the Input Data

Air quality measurements are fetched from the **OpenAQ v3 API**. An API key is required and can be obtained for free at [https://explore.openaq.org/register](https://explore.openaq.org/register). The key must be set as the `OPENAQ_API_KEY` environment variable.

Alternatively, data can be sourced from **SAGE** edge sensors using the `sage_data_client` library or a JSONL data file.

## Workflow

The workflow processes air quality data through two parallel pipelines after initial extraction:

| Job Label              | Pipeline  | Description                                                              |
|------------------------|-----------|--------------------------------------------------------------------------|
| extract_timeseries     | Shared    | Extracts AQI time series from raw measurement data                       |
| analyze_pollutants     | Base      | Creates visualizations and statistical summaries for each pollutant      |
| detect_anomalies       | Base      | Detects statistical outliers, sustained high AQI, and sudden spikes      |
| merge                  | Base      | Merges anomaly results across multiple locations                         |
| fetch_historical       | Forecast  | Fetches 90 days of historical data from OpenAQ for LSTM training         |
| prepare_features       | Forecast  | Creates temporal/statistical features and LSTM input sequences           |
| train_model            | Forecast  | Trains a multi-layer LSTM model with early stopping                      |
| generate_forecast      | Forecast  | Generates 24-hour AQI predictions with confidence intervals              |
| visualize_forecast     | Forecast  | Creates forecast plots with historical context and AQI category bands    |

In [ ]:
!pip3 install pandas

## 1. Create the Air Quality Forecast Workflow

First, configure the monitoring location(s) and date range. You can use `fetch_openaq_catalog.py --search` to find location IDs.

Then the workflow class below will:
1. Fetch measurement data from the OpenAQ API (or SAGE sensors)
2. Build the Pegasus catalogs (sites, transformations, replicas)
3. Construct the DAG with both base and forecast pipelines

In [ ]:
# Update the OpenAQ location IDs to analyze
# Use: ./fetch_openaq_catalog.py --search --city "Vienna" to find IDs
LOCATION_IDS = [4603, 2178, 1490]

# Update the start and end dates for data extraction
START_DATE = '2024-01-15'
# Default end date is set as +1 day
END_DATE = None

# Parameters to analyze (choices: pm25, pm10, o3, no2, so2, co)
PARAMETERS = ['pm25', 'pm10', 'o3', 'no2', 'so2', 'co']

# Number of days of historical data for LSTM training
HISTORICAL_DAYS = 90

# Forecast horizon in hours
FORECAST_HORIZON = 24

### OpenAQ API Key

An API key is required to fetch air quality data from the OpenAQ v3 API. You can register for a free key at [https://explore.openaq.org/register](https://explore.openaq.org/register). Replace `YOUR_API_KEY_HERE` below with your actual key.

In [ ]:
import os

# Set your OpenAQ API key (get one free at https://explore.openaq.org/register)
os.environ["OPENAQ_API_KEY"] = "YOUR_API_KEY_HERE"

**Note:** Historical data of 90 days is recommended for good model accuracy. Reduce to 30 days for faster testing.

## Build the Apptainer container

The workflow runs every job inside `Apptainer/AirQuality_Forecast_Container.sif`. Build it once here.
`Bootstrap: docker` in the definition file makes Apptainer pull and convert the
base image itself, so **no Docker installation and no registry login are needed**,
and Pegasus stages the resulting `.sif` to the worker nodes (`image_site="local"`).

A `.sif` carries a single architecture, so build it on this cluster rather than
copying one from a laptop. Skip this cell if the image already exists.


In [ ]:
# Build only if missing -- the build takes several minutes.
!test -f Apptainer/AirQuality_Forecast_Container.sif || apptainer build Apptainer/AirQuality_Forecast_Container.sif Apptainer/AirQuality_Forecast_Container.def

# Confirm the image is usable and has what PegasusLite needs.
!apptainer exec Apptainer/AirQuality_Forecast_Container.sif which curl wget
!apptainer exec Apptainer/AirQuality_Forecast_Container.sif python -c "import torch, sklearn, pandas; print('deps ok')"


In [ ]:
import os
import sys
import logging
from pathlib import Path
from datetime import datetime, timedelta

# --- Import Pegasus API ---
from Pegasus.api import *
logging.basicConfig(level=logging.DEBUG)
logging.basicConfig()


# --- Main workflow class ---
class AirQualityForecastWorkflow():
    wf = None
    sc = None
    tc = None
    rc = None
    props = None

    dagfile = None
    wf_dir = None
    shared_scratch_dir = None
    local_storage_dir = None
    wf_name = "airquality_forecast"

    openaq_catalog = None
    openaq_cache_file = "openaq_catalog.csv"

    # --- Init ---
    def __init__(self, location_ids, start_date, end_date, parameters,
                 historical_days=90, forecast_horizon=24,
                 dagfile="workflow.yml"):
        self.dagfile = dagfile
        self.wf_dir = str(Path(".").resolve())
        self.shared_scratch_dir = os.path.join(self.wf_dir, "scratch")
        self.local_storage_dir = os.path.join(self.wf_dir, "output")
        self.location_ids = location_ids
        self.parameters = parameters
        self.start_date = start_date
        self.end_date = end_date
        self.historical_days = historical_days
        self.forecast_horizon = forecast_horizon
        self.historical_start_date = start_date - timedelta(days=historical_days)

    # --- Write files in directory ---
    def write(self):
        if not self.sc is None:
            self.sc.write()
        self.props.write()
        self.rc.write()
        self.tc.write()

        try:
            self.wf.write(file=self.dagfile)
        except PegasusClientError as e:
            print(e)

    # --- Plan and Submit the workflow ---
    def plan_submit(self):
        try:
            self.wf.plan(submit=True)
        except PegasusClientError as e:
            print(e)

    # --- Get status of the workflow ---
    def status(self):
        try:
            self.wf.status(long=True)
        except PegasusClientError as e:
            print(e)
            
    # --- Wait for the workflow to finish -----------------------------------------------
    def wait(self):
        try:
            self.wf.wait()
        except PegasusClientError as e:
            print(e)            

    # --- Get statistics of the workflow ---
    def statistics(self):
        try:
            self.wf.statistics()
        except PegasusClientError as e:
            print(e)

    # --- Configuration (Pegasus Properties) ---
    def create_pegasus_properties(self):
        self.props = Properties()
        self.props["pegasus.transfer.threads"] = "16"
        return

    # --- Site Catalog ---
    def create_sites_catalog(self, exec_site_name="condorpool"):
        self.sc = SiteCatalog()

        local = (Site("local")
                    .add_directories(
                        Directory(Directory.SHARED_SCRATCH, self.shared_scratch_dir)
                            .add_file_servers(FileServer("file://" + self.shared_scratch_dir, Operation.ALL)),
                        Directory(Directory.LOCAL_STORAGE, self.local_storage_dir)
                            .add_file_servers(FileServer("file://" + self.local_storage_dir, Operation.ALL))
                    )
                )

        exec_site = (Site(exec_site_name)
                        .add_condor_profile(universe="vanilla")
                        .add_pegasus_profile(style="condor")
                    )

        self.sc.add_sites(local, exec_site)

    # --- Transformation Catalog (Executables and Containers) ---
    def create_transformation_catalog(self, exec_site_name="condorpool"):
        self.tc = TransformationCatalog()

        # Both containers are backed by the same locally built Apptainer image.
        # Pegasus stages the .sif like any other input file, so image_site is
        # "local" -- the site where the file physically lives.
        sif = os.path.join(self.wf_dir, "Apptainer/AirQuality_Forecast_Container.sif")
        if not os.path.exists(sif):
            print(f"Warning: {sif} not found -- run the build cell above first")
        image_url = "file://" + sif

        # Base workflow container
        airquality_container = Container("airquality_container",
            container_type=Container.SINGULARITY,
            image=image_url,
            image_site="local"
        )

        # Forecast workflow container (with PyTorch)
        forecast_container = Container("airquality_forecast_container",
            container_type=Container.SINGULARITY,
            image=image_url,
            image_site="local"
        )

        # Base transformations
        mkdir = Transformation("mkdir", site="local", pfn="/bin/mkdir", is_stageable=False)

        extract_timeseries = Transformation("extract_timeseries", site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/extract_aqi_timeseries.py"),
            is_stageable=True, container=airquality_container
        ).add_pegasus_profile(memory="2 GB")

        analyze_pollutants = Transformation("analyze_pollutants", site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/analyze_pollutants.py"),
            is_stageable=True, container=airquality_container
        ).add_pegasus_profile(memory="2 GB")

        detect_anomalies = Transformation("detect_anomalies", site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/detect_anomalies.py"),
            is_stageable=True, container=airquality_container
        ).add_pegasus_profile(memory="1 GB")

        merge = Transformation("merge", site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/merge.py"),
            is_stageable=True, container=airquality_container
        ).add_pegasus_profile(memory="1 GB")

        # Forecast transformations
        fetch_historical = Transformation("fetch_historical", site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/fetch_historical_data.py"),
            is_stageable=True, container=forecast_container
        ).add_pegasus_profile(memory="2 GB")

        prepare_features = Transformation("prepare_features", site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/prepare_features.py"),
            is_stageable=True, container=forecast_container
        ).add_pegasus_profile(memory="2 GB")

        train_model = Transformation("train_model", site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/train_forecast_model.py"),
            is_stageable=True, container=forecast_container
        ).add_pegasus_profile(memory="4 GB")

        generate_forecast = Transformation("generate_forecast", site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/generate_forecast.py"),
            is_stageable=True, container=forecast_container
        ).add_pegasus_profile(memory="2 GB")

        visualize_forecast = Transformation("visualize_forecast", site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/visualize_forecast.py"),
            is_stageable=True, container=forecast_container
        ).add_pegasus_profile(memory="2 GB")

        self.tc.add_containers(airquality_container, forecast_container)
        self.tc.add_transformations(
            mkdir, extract_timeseries, analyze_pollutants, detect_anomalies, merge,
            fetch_historical, prepare_features, train_model, generate_forecast, visualize_forecast
        )

    # --- Fetch OpenAQ catalog ---
    def fetch_openaq_catalog(self):
        print("Fetching OpenAQ data...")

        sys.path.insert(0, self.wf_dir)
        from fetch_openaq_catalog import fetch_openaq_catalog, save_catalog

        df = fetch_openaq_catalog(
            location_ids=self.location_ids,
            start_date=self.start_date,
            end_date=self.end_date,
            parameters=self.parameters
        )

        if df.empty:
            print("No data fetched from OpenAQ")
            return False

        save_catalog(df, self.openaq_cache_file)
        self.openaq_catalog = df
        return True

    # --- Replica Catalog ---
    def create_replica_catalog(self):
        self.rc = ReplicaCatalog()

        if self.openaq_catalog is None:
            if not self.fetch_openaq_catalog():
                print("Failed to fetch OpenAQ data")
                sys.exit(1)

        self.rc.add_replica(
            "local",
            "openaq_catalog.csv",
            "file://" + os.path.join(self.wf_dir, self.openaq_cache_file)
        )

    # --- Create Workflow ---
    def create_workflow(self):
        self.wf = Workflow(self.wf_name, infer_dependencies=True)

        catalog_file = File("openaq_catalog.csv")

        if self.openaq_catalog is None or self.openaq_catalog.empty:
            print("Error: No catalog data available. Run fetch_openaq_catalog first.")
            return

        # Get unique location names for each location ID
        location_map = {}
        for loc_id in self.location_ids:
            loc_data = self.openaq_catalog[self.openaq_catalog['location_id'] == loc_id]
            if not loc_data.empty:
                loc_name = loc_data['location'].iloc[0]
                safe_name = loc_name.replace(' ', '_').replace('-', '_').replace('/', '_')
                location_map[loc_id] = {
                    'name': safe_name,
                    'display_name': loc_name
                }

        print(f"\nCreating workflow for {len(location_map)} location(s)")
        print(f"Historical data period: {self.historical_days} days")
        print(f"Forecast horizon: {self.forecast_horizon} hours\n")

        anomaly_files = []

        for loc_id, loc_info in location_map.items():
            location = loc_info['name']
            display_name = loc_info['display_name']

            print(f"  Processing location: {display_name} (ID: {loc_id})")

            # Create directories
            mkdir_job = (
                Job("mkdir", _id=f"mkdir_{location}", node_label=f"mkdir_{location}")
                .add_args(
                    f"-p {self.local_storage_dir}/timeseries/{location} "
                    f"{self.local_storage_dir}/analysis/{location} "
                    f"{self.local_storage_dir}/anomalies/{location} "
                    f"{self.local_storage_dir}/historical/{location} "
                    f"{self.local_storage_dir}/features/{location} "
                    f"{self.local_storage_dir}/models/{location} "
                    f"{self.local_storage_dir}/forecasts/{location}"
                )
                .add_profiles(Namespace.SELECTOR, key="execution.site", value="local")
            )
            self.wf.add_jobs(mkdir_job)

            # ===== BASE PIPELINE =====

            # Extract time series (shared by both pipelines)
            timeseries_file = File(f"timeseries/{location}/{location}_timeseries.json")
            extract_job = (
                Job("extract_timeseries", _id=f"extract_{location}", node_label=f"extract_{location}")
                .add_args(f"-i openaq_catalog.csv -o timeseries/{location}")
                .add_inputs(catalog_file)
                .add_outputs(timeseries_file, stage_out=False, register_replica=False)
                .add_pegasus_profiles(label=location)
            )
            self.wf.add_jobs(extract_job)
            self.wf.add_dependency(mkdir_job, children=[extract_job])

            # Analyze pollutants
            analysis_png = File(f"analysis/{location}/{location}_analysis.png")
            stats_file = File(f"analysis/{location}/{location}_statistics.json")
            analyze_job = (
                Job("analyze_pollutants", _id=f"analyze_{location}", node_label=f"analyze_{location}")
                .add_args(f"-i timeseries/{location}/{location}_timeseries.json -o analysis/{location}")
                .add_inputs(timeseries_file)
                .add_outputs(analysis_png, stats_file, stage_out=True, register_replica=False)
                .add_pegasus_profiles(label=location)
            )
            self.wf.add_jobs(analyze_job)

            # Detect anomalies
            anomaly_file = File(f"anomalies/{location}/{location}_anomalies.json")
            anomaly_files.append(anomaly_file)
            anomaly_job = (
                Job("detect_anomalies", _id=f"anomaly_{location}", node_label=f"anomaly_{location}")
                .add_args(
                    f"-i timeseries/{location}/{location}_timeseries.json "
                    f"-o anomalies/{location}/{location}_anomalies.json -t 3.0"
                )
                .add_inputs(timeseries_file)
                .add_outputs(anomaly_file, stage_out=True, register_replica=False)
                .add_pegasus_profiles(label=location)
            )
            self.wf.add_jobs(anomaly_job)

            # ===== FORECAST PIPELINE =====

            # Fetch historical data (90 days)
            historical_file = File(f"historical/{location}/{location}_historical.csv")
            fetch_hist_job = (
                Job("fetch_historical", _id=f"fetch_hist_{location}", node_label=f"fetch_hist_{location}")
                .add_args(
                    f"--location-id {loc_id} "
                    f"--days {self.historical_days} "
                    f"--end-date {self.start_date.strftime('%Y-%m-%d')} "
                    f"--output historical/{location}/{location}_historical.csv"
                )
                .add_outputs(historical_file, stage_out=False, register_replica=False)
                .add_env(OPENAQ_API_KEY=os.environ.get('OPENAQ_API_KEY', ''))
                .add_pegasus_profiles(label=f"{location}_forecast")
            )
            self.wf.add_jobs(fetch_hist_job)
            self.wf.add_dependency(mkdir_job, children=[fetch_hist_job])

            # Prepare features (depends on both timeseries and historical data)
            features_file = File(f"features/{location}/{location}_train.npz")
            scaler_file = File(f"features/{location}/{location}_train_scaler.json")
            prepare_job = (
                Job("prepare_features", _id=f"prepare_{location}", node_label=f"prepare_{location}")
                .add_args(
                    f"--timeseries timeseries/{location}/{location}_timeseries.json "
                    f"--historical historical/{location}/{location}_historical.csv "
                    f"--output features/{location}/{location}_train.npz "
                    f"--lookback 168 "
                    f"--horizon {self.forecast_horizon}"
                )
                .add_inputs(timeseries_file, historical_file)
                .add_outputs(features_file, scaler_file, stage_out=False, register_replica=False)
                .add_pegasus_profiles(label=f"{location}_forecast")
            )
            self.wf.add_jobs(prepare_job)
            self.wf.add_dependency(extract_job, children=[prepare_job])
            self.wf.add_dependency(fetch_hist_job, children=[prepare_job])

            # Train LSTM model
            model_checkpoint = File(f"models/{location}/{location}_lstm_checkpoint.pt")
            training_info = File(f"models/{location}/{location}_training_info.json")
            train_job = (
                Job("train_model", _id=f"train_{location}", node_label=f"train_{location}")
                .add_args(
                    f"--features features/{location}/{location}_train.npz "
                    f"--output models/{location} "
                    f"--location-name {location} "
                    f"--epochs 100 "
                    f"--batch-size 32 "
                    f"--patience 10"
                )
                .add_inputs(features_file, scaler_file)
                .add_outputs(model_checkpoint, training_info, stage_out=True, register_replica=False)
                .add_pegasus_profiles(label=f"{location}_forecast")
            )
            self.wf.add_jobs(train_job)
            self.wf.add_dependency(prepare_job, children=[train_job])

            # Generate forecast
            forecast_file = File(f"forecasts/{location}/{location}_forecast.json")
            forecast_job = (
                Job("generate_forecast", _id=f"forecast_{location}", node_label=f"forecast_{location}")
                .add_args(
                    f"--model models/{location}/{location}_lstm_checkpoint.pt "
                    f"--timeseries timeseries/{location}/{location}_timeseries.json "
                    f"--scaler features/{location}/{location}_train_scaler.json "
                    f"--output forecasts/{location}/{location}_forecast.json "
                    f'--location-name \"{display_name}\" '
                    f"--lookback 168"
                )
                .add_inputs(model_checkpoint, timeseries_file, scaler_file)
                .add_outputs(forecast_file, stage_out=True, register_replica=False)
                .add_pegasus_profiles(label=f"{location}_forecast")
            )
            self.wf.add_jobs(forecast_job)
            self.wf.add_dependency(train_job, children=[forecast_job])

            # Visualize forecast
            forecast_viz = File(f"forecasts/{location}/{location}_forecast.png")
            forecast_summary = File(f"forecasts/{location}/{location}_forecast_summary.json")
            viz_job = (
                Job("visualize_forecast", _id=f"viz_forecast_{location}", node_label=f"viz_forecast_{location}")
                .add_args(
                    f"--timeseries timeseries/{location}/{location}_timeseries.json "
                    f"--forecast forecasts/{location}/{location}_forecast.json "
                    f"--output forecasts/{location}/{location}_forecast.png "
                    f"--lookback-days 7"
                )
                .add_inputs(timeseries_file, forecast_file)
                .add_outputs(forecast_viz, forecast_summary, stage_out=True, register_replica=False)
                .add_pegasus_profiles(label=f"{location}_forecast")
            )
            self.wf.add_jobs(viz_job)
            self.wf.add_dependency(forecast_job, children=[viz_job])

        # Merge all anomaly results (base workflow final step)
        if len(anomaly_files) > 1:
            merged_anomalies = File("merged_anomalies.json")
            merge_job = (
                Job("merge", _id="merge_all_anomalies", node_label="merge_all")
                .add_args(f"-i {' '.join([f.lfn for f in anomaly_files])} -o {merged_anomalies.lfn}")
                .add_inputs(*anomaly_files)
                .add_outputs(merged_anomalies, stage_out=True, register_replica=False)
            )
            self.wf.add_jobs(merge_job)


# --- Build and generate the workflow ---
start_date = datetime.strptime(START_DATE, '%Y-%m-%d')
if END_DATE:
    end_date = datetime.strptime(END_DATE, '%Y-%m-%d')
else:
    end_date = start_date + timedelta(days=1)

dagfile = 'workflow.yml'

workflow = AirQualityForecastWorkflow(
    location_ids=LOCATION_IDS,
    start_date=start_date,
    end_date=end_date,
    parameters=PARAMETERS,
    historical_days=HISTORICAL_DAYS,
    forecast_horizon=FORECAST_HORIZON,
    dagfile=dagfile
)

print("Creating execution sites...")
workflow.create_sites_catalog("condorpool")

print("Creating workflow properties...")
workflow.create_pegasus_properties()

print("Creating transformation catalog...")
workflow.create_transformation_catalog("condorpool")

print("Creating replica catalog...")
workflow.create_replica_catalog()

print("Creating air quality forecast workflow DAG...")
workflow.create_workflow()

workflow.write()
print("\nAir Quality Forecast Workflow has been generated!")

## View the Generated Workflow DAG

Before submitting, we can visualize the workflow DAG using `pegasus-graphviz`. The graph shows all jobs as nodes and data dependencies as edges. For a single location you should see the two parallel pipelines (base and forecast) branching from the shared `extract_timeseries` job.

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image
Image(filename='workflow.png')

## 2. Plan and Submit the Workflow

We will now plan and submit the workflow for execution. By default we are running jobs on site **condorpool** i.e. the selected ACCESS resource.

In [ ]:
workflow.plan_submit()

After the workflow has been successfully planned and submitted, you can use the Python `Workflow` object to monitor the status of the workflow. It shows in detail the counts of jobs of each status and whether a job is idle or running.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 3. Statistics

Depending on whether the workflow finished successfully or not, you have options on what to do next. If the workflow failed you can use `workflow.analyze()` to get help finding out what went wrong. If the workflow finished successfully, we can pull out some statistics from the provenance database:

In [ ]:
workflow.statistics()

## 4. Examining the Results

Once the workflow has finished, we can look at the output directory for our results. The workflow produces the following outputs for each location:

```
output/
├── analysis/<location>/
│   ├── <location>_analysis.png           # Pollutant concentration plots with AQI bands
│   └── <location>_statistics.json         # Statistical summary per parameter
├── anomalies/<location>/
│   └── <location>_anomalies.json         # Detected outliers, sustained alerts, spikes
├── models/<location>/
│   ├── <location>_lstm_checkpoint.pt      # Trained LSTM model weights
│   └── <location>_training_info.json      # Training history and metrics
├── forecasts/<location>/
│   ├── <location>_forecast.json           # 24-hour AQI predictions with confidence intervals
│   ├── <location>_forecast.png            # Forecast visualization with historical context
│   └── <location>_forecast_summary.json   # Forecast statistics and category distribution
└── merged_anomalies.json                  # Merged anomalies (multi-location only)
```

In [ ]:
!ls -ltR output/

### Pollutant Analysis

The analysis visualization shows concentration levels over time for each pollutant, with AQI category color bands in the background (green = Good, yellow = Moderate, orange = Unhealthy for Sensitive Groups, etc.).

In [ ]:
import glob
from IPython.display import Image, display

analysis_pngs = sorted(glob.glob("images/analysis/**/*_analysis.png", recursive=True))
for png in analysis_pngs:
    print(f"\n{png}")
    display(Image(filename=png))

### AQI Forecast

The forecast visualization shows historical AQI values (blue) alongside the 24-hour LSTM forecast (orange dashed) with a 95% confidence interval shaded region. AQI category bands are shown on the right axis.

In [ ]:
forecast_pngs = sorted(glob.glob("images/forecasts/**/*_forecast.png", recursive=True))
for png in forecast_pngs:
    print(f"\n{png}")
    display(Image(filename=png))

### Anomaly Detection Summary

The anomaly detection results include statistical outliers (Z-score > 3.0), sustained high AQI periods (AQI >= 150 for 3+ consecutive hours), and sudden concentration spikes (2x hour-over-hour increase).

In [ ]:
import json

anomaly_files = sorted(glob.glob("images/anomalies/**/*_anomalies.json", recursive=True))
for anomaly_file in anomaly_files:
    with open(anomaly_file, 'r') as f:
        anomaly_data = json.load(f)

    location = anomaly_data.get('location', 'Unknown')
    summary = anomaly_data.get('summary', {})

    print(f"\n--- {location} ---")
    print(f"  Statistical outliers:    {summary.get('total_outliers', 0)}")
    print(f"  Sustained high AQI:      {summary.get('total_sustained_alerts', 0)}")
    print(f"  Sudden spikes:           {summary.get('total_spikes', 0)}")
    print(f"  Parameters affected:     {summary.get('parameters_affected', [])}")

### Forecast Summary

The forecast summary shows predicted AQI statistics and the distribution of forecast categories over the prediction horizon.

In [ ]:
summary_files = sorted(glob.glob("images/forecasts/**/*_forecast_summary.json", recursive=True))
for summary_file in summary_files:
    with open(summary_file, 'r') as f:
        summary_data = json.load(f)

    location = summary_data.get('location', 'Unknown')
    stats = summary_data.get('forecast_statistics', {})
    categories = summary_data.get('category_distribution', {})

    print(f"\n--- {location} ---")
    print(f"  Forecast period: {summary_data.get('forecast_period', {}).get('start', 'N/A')} "
          f"to {summary_data.get('forecast_period', {}).get('end', 'N/A')}")
    print(f"  Mean AQI:  {stats.get('mean_aqi', 'N/A'):.1f}")
    print(f"  Min AQI:   {stats.get('min_aqi', 'N/A'):.1f}")
    print(f"  Max AQI:   {stats.get('max_aqi', 'N/A'):.1f}")
    print(f"  Std AQI:   {stats.get('std_aqi', 'N/A'):.1f}")
    print(f"  Category distribution:")
    for category, count in categories.items():
        print(f"    {category}: {count} hours")